In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

import json
import torch
from torch.utils.data import Dataset
from transformers import DebertaV2Tokenizer

class ORDataset(Dataset):
    
    def __init__(self, json_path, tokenizer, max_length=512, mode='train'):
        """
        Args:
            json_path (str): 数据文件路径.
            tokenizer (DebertaV2Tokenizer): 分词器.
            max_length (int): 最大序列长度.
            mode (str): 数据集模式 ('train', 'eval', 'predict').
        """
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.mode = mode
        self.samples = []
        
        # 1. 读取JSON
        with open(json_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        # 2. 构造样本
        for key, item in data.items():
            
            homonym = item["homonym"]
            definition = item["judged_meaning"]
            example = item["example_sentence"]
            context = f"{item['precontext']} {item['sentence']} {item['ending']}"
            
            target_avg = item.get("average", 0.0)
            target_stdev = item.get("stdev", 0.0)
            target_score = round(target_avg) 
            target_score = max(1, min(5, target_score)) # 确保在 1-5 范围内
            
            self.samples.append({
                "json_key": key,
                "homonym": homonym,
                "definition": definition,
                "example": example,
                "context": context,
                # 0 到 4 的整数标签
                "ordinal_label": target_score - 1, 
                "target_avg": target_avg,    
                "target_stdev": target_stdev,  
                "sample_id": item['sample_id']
            })
            
        print(f"✅ 创建了 {len(self.samples)} 个有序分类样本 (Mode: {self.mode})")
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        sample = self.samples[idx]
        
        # 1. 构造输入文本
        text_parts = (
            f"homonym：{sample['homonym']}",
            f"Definition:{sample['definition']}",
            f"Example:{sample['example']}",
            f"Context:{sample['context']}"
        )
        text = self.tokenizer.sep_token.join(text_parts)
        
        # 2. Tokenize
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt" 
        )
        
        # 移除batch维度
        encoding = {k: v.squeeze(0) for k, v in encoding.items()}
        
        # 移除 token_type_ids
        if "token_type_ids" in encoding:
            del encoding["token_type_ids"]
            
        # 3. 标签分配：根据模式返回不同的标签
        
        # ID 作为字符串返回，需要 DataLoader/DataCollator 特殊处理
        encoding["id"] = sample["json_key"] 
        
        if self.mode in ['train', 'eval']:
            
            encoding["labels"] = torch.tensor(sample["ordinal_label"], dtype=torch.long)
            
            # 额外的回归和标准差，用于自定义加权损失（如果未使用，Trainer 会忽略）
            encoding["avg_raw"] = torch.tensor(sample["target_avg"], dtype=torch.float32)
            encoding["stdev_raw"] = torch.tensor(sample["target_stdev"], dtype=torch.float32)
            
        elif self.mode == 'predict':
            # 不返回 'labels' 或其他标签，避免 Trainer/DataLoader 尝试将其转换为 None/Tensor
            pass
            
        return encoding

In [ ]:
import torch.nn as nn
from transformers import DebertaV2Model, DebertaV2PreTrainedModel
import torch

import torch
import torch.nn as nn
from torch.nn.init import uniform_
import torch.nn.functional as F
from transformers import DebertaV2Model, DebertaV2PreTrainedModel

# 损失函数
class OrdinalRegressionLoss(nn.Module):
    """
    基于累积 Logit 的序数回归损失函数（负对数似然）。
    接受有序的截止点和潜变量 Y*。
    """
    def __init__(self, num_class=5, train_cutpoints=False, scale=20.0):
        super().__init__()
        self.num_classes = num_class
        
        # 分割点 = 类别 - 1
        num_cutpoints = self.num_classes - 1
        
        # 初始化切点：均匀分布在[-scale/2, scale/2]区间
        # self.cutpoints = torch.arange(num_cutpoints).float()*scale/(num_class-2) - scale / 2
        initial_cutpoints = torch.arange(num_cutpoints).float() * scale / (num_class - 2) - scale / 2
        # initial_cutpoints = torch.tensor([0.0, 0.3, 0.5, 0.7], dtype=torch.float32)
        
        # 转换成可训练张量
        # self.cutpoints = nn.Parameter(self.cutpoints)
        
        self.register_buffer('cutpoints', initial_cutpoints)

        # print("debug1:", self.cutpoints.data)
        
        # 切点训练开关
        if not train_cutpoints:
            self.cutpoints.requires_grad_(False)

    def forward(self, pred, labels):
        
        sigmoids = torch.sigmoid(self.cutpoints - pred)
        # print("sigmoids:", sigmoids)
        
        # P(Y=k) = P(Y<=k) - P(Y<=k-1)
        link_mat = sigmoids[:, 1:] - sigmoids[:, :-1]
        
        link_mat = torch.cat((
                sigmoids[:, [0]],    # P(类别=0) = σ(c0-pred)
                link_mat,    # P(类别=1,2,3)
                (1 - sigmoids[:, [-1]])    # P(类别=4) = 1 - σ(c3-pred)
            ),
            dim=1
        )
        
        eps = 1e-15
        likelihoods = torch.clamp(link_mat, eps, 1 - eps)  # 防止log(0)
        
        neg_log_likelihood = torch.log(likelihoods)  # 对数似然
        
        if labels is None:
            loss = 0
        else:
            loss = -torch.gather(neg_log_likelihood, 1, labels).mean()
            
        return loss, likelihoods


class DebertaV2ForOrdinalRegression(DebertaV2PreTrainedModel):
    
    def __init__(self, config):
        super().__init__(config)
        
        self.deberta = DebertaV2Model(config)
        
        self.regressor = nn.Sequential(
            nn.Linear(config.hidden_size, config.hidden_size),
            nn.GELU(),
            nn.LayerNorm(config.hidden_size),
            nn.Linear(config.hidden_size, 1),
            # nn.Tanh()
        )
        
        # 有序回归损失函数
        self.ordinal_loss = OrdinalRegressionLoss(
            num_class=5,    # 1-5分，共5个类别
            train_cutpoints=False,  # 学习切点
            scale=6.0              # 主要调整参数
        )
        
        self.init_weights()
        
    def get_cutpoints(self):
        """获取当前的切点值"""
        return self.ordinal_loss.cutpoints.data.cpu().numpy()

    # -----------------------------------------------------y_star-----------------

    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
        
        # print("Current cutpoints:", self.ordinal_loss.cutpoints.data)
        # print("Requires grad:", self.ordinal_loss.cutpoints.requires_grad)
        # 编码
        outputs = self.deberta(input_ids=input_ids, attention_mask=attention_mask)
        
        X = outputs[0][:, 0, :] 
        
        # 1. 预测潜变量 Y*
        y_star = self.regressor(X) # 形状: [B, 1]


        y_star_flat = y_star.squeeze(-1).detach()
        y_star_mean = y_star_flat.mean().item()
        y_star_std = y_star_flat.std().item()

        print("--- Y* Stats ---")
        print(f"Y* Mean: {y_star_mean:.4f}")
        print(f"Y* Std. Dev.: {y_star_std:.4f}")
        print("----------------")
        
        # print("y_star:", y_star)
        # 2. 损失计算
        if labels is not None:
            # 确保labels形状正确 [batch_size, 1]
            if labels.dim() == 1:
                labels = labels.unsqueeze(-1)
            
            loss, probs = self.ordinal_loss(y_star, labels)
        else:
            loss = None
            _, probs = self.ordinal_loss(y_star, None)
             
        return {
            "loss": loss,
            "logits": probs,           # 类别概率
            "y_star": y_star           # 潜变量预测
        }
    
    def predict_score(self, input_ids, attention_mask=None):
        """预测1-5分"""
        with torch.no_grad():
            outputs = self(input_ids=input_ids, attention_mask=attention_mask)
            probs = outputs["logits"]  # [batch_size, 5]
            pred_class = torch.argmax(probs, dim=-1)  # 0-4
            return pred_class + 1  # 转换为1-5分

2025-12-10 05:39:17.315356: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765345157.534441      47 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765345157.606366      47 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [ ]:
import os
os.environ["WANDB_MODE"] = "disabled"

from transformers import DebertaV2Tokenizer, DebertaV2Config, Trainer, TrainingArguments

# from model import DebertaV2ForOrdinalRegression
# from data_load import ORDataset

MODEL_NAME = "/kaggle/input/semeval/deberta-v3-large" 
TRAIN_JSON_PATH = "/kaggle/input/semeval/data/train.json" 
TEST_JSON_PATH = "/kaggle/input/semeval/data/dev.json" # 使用 dev.json 进行验证
OUTPUT_DIR = "/kaggle/working/output_ordinal_regression"

if __name__ == "__main__":
    
    # 加载 Tokenizer 和配置
    tokenizer = DebertaV2Tokenizer.from_pretrained(MODEL_NAME)
    config = DebertaV2Config.from_pretrained(MODEL_NAME)
    
    # 实例化序数回归模型
    # model = DebertaV2ForOrdinalRegression.from_pretrained(config=config)
    model = DebertaV2ForOrdinalRegression.from_pretrained(MODEL_NAME, config=config)


    cutpoints = model.get_cutpoints()
    print("Current cutpoints:", cutpoints)

    # 实例化数据集
    train_dataset = ORDataset(
        json_path=TRAIN_JSON_PATH,
        tokenizer=tokenizer,
        mode='train',
        max_length=256
        )

    # 训练参数配置
    training_args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        num_train_epochs=3,                       # 增加 epochs
        per_device_train_batch_size=8,
        gradient_accumulation_steps=4,
        
        save_strategy="no",                    # no
        
        warmup_steps=500,                       
        weight_decay=0.01,                      
        logging_dir='./logs_ordinal',       
        logging_steps=50,                       
        learning_rate=1e-5,                     
        fp16=True,                              
        seed=42,
        # remove_unused_columns=False,                   
    )

    # 实例化 Trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        tokenizer=tokenizer,
    )

    # 启动训练
    print("***** 开始微调 DeBERTaV2 序数回归模型 *****")
    trainer.train()

    # 训练结束后，保存最终模型
    trainer.save_model(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)
    print(f"训练完成，最终最佳模型已保存至 {OUTPUT_DIR}")

Some weights of DebertaV2ForOrdinalRegression were not initialized from the model checkpoint at /kaggle/input/semeval/deberta-v3-large and are newly initialized: ['ordinal_loss.cutpoints', 'regressor.0.bias', 'regressor.0.weight', 'regressor.2.bias', 'regressor.2.weight', 'regressor.3.bias', 'regressor.3.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_47/2502431142.py:63: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Current cutpoints: [-3. -1.  1.  3.]
✅ 创建了 2280 个有序分类样本 (Mode: train)


/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

***** 开始微调 DeBERTaV2 序数回归模型 *****
--- Y* Stats ---
Y* Mean: 0.4722
Y* Std. Dev.: 0.1899
----------------
--- Y* Stats ---
Y* Mean: 0.5757
Y* Std. Dev.: 0.2389
----------------
--- Y* Stats ---
Y* Mean: 0.7158
Y* Std. Dev.: 0.1548
----------------
--- Y* Stats ---
Y* Mean: 0.7690
Y* Std. Dev.: 0.1912
----------------


Step,Training Loss
50,7.076600
100,6.857000
150,6.839700
200,6.849100


--- Y* Stats ---
Y* Mean: 0.6431
Y* Std. Dev.: 0.1653
----------------
--- Y* Stats ---
Y* Mean: 0.5117
Y* Std. Dev.: 0.1611
----------------
--- Y* Stats ---
Y* Mean: 0.6064
Y* Std. Dev.: 0.2399
----------------
--- Y* Stats ---
Y* Mean: 0.6909
Y* Std. Dev.: 0.2410
----------------
--- Y* Stats ---
Y* Mean: 0.6294
Y* Std. Dev.: 0.2759
----------------
--- Y* Stats ---
Y* Mean: 0.5449
Y* Std. Dev.: 0.1670
----------------
--- Y* Stats ---
Y* Mean: 0.5698
Y* Std. Dev.: 0.1610
----------------
--- Y* Stats ---
Y* Mean: 0.6055
Y* Std. Dev.: 0.2230
----------------
--- Y* Stats ---
Y* Mean: 0.5742
Y* Std. Dev.: 0.1313
----------------
--- Y* Stats ---
Y* Mean: 0.6963
Y* Std. Dev.: 0.2440
----------------
--- Y* Stats ---
Y* Mean: 0.6411
Y* Std. Dev.: 0.2366
----------------
--- Y* Stats ---
Y* Mean: 0.6099
Y* Std. Dev.: 0.2830
----------------
--- Y* Stats ---
Y* Mean: 0.7129
Y* Std. Dev.: 0.2605
----------------
--- Y* Stats ---
Y* Mean: 0.6172
Y* Std. Dev.: 0.2155
----------------
--- Y*

In [ ]:
import os
import json
import torch
import numpy as np
from torch.utils.data import DataLoader, SequentialSampler
from tqdm import tqdm

TEST_JSON_PATH = "/kaggle/input/semeval/data/dev.json" 
INFERENCE_BATCH_SIZE = 32
OUTPUT_DIR = "./output_ordinal_regression"
os.makedirs(OUTPUT_DIR, exist_ok=True)
OUTPUT_JSONL_PATH = os.path.join(OUTPUT_DIR, "dev_predictions_mean.jsonl")

device = model.device 
model.eval() 

print(f"Model is on: {device}")
print(f"Using Test/Inference file: {TEST_JSON_PATH}")

dev_dataset = ORDataset(
    json_path=TEST_JSON_PATH, 
    tokenizer=tokenizer,
    max_length=256,
    mode='predict',
)

dev_dataloader = DataLoader(
    dev_dataset,
    sampler=SequentialSampler(dev_dataset),
    batch_size=INFERENCE_BATCH_SIZE,
)


all_results = []
# 类别值 (1-5) 权重
score_tensor = torch.tensor([1, 2, 3, 4, 5], dtype=torch.float32, device=device).unsqueeze(0)

print("\n***** 开始批量手动推理 (期望值预测) *****")

with torch.no_grad():
    for batch in tqdm(dev_dataloader, desc="Inferencing"):
        
        # 提取输入和 ID
        ids = batch.pop("id") 
        
        # 运行模型
        outputs = model(
            input_ids=batch["input_ids"].to(device),
            attention_mask=batch["attention_mask"].to(device)
        )
        
        # 提取预测值：likelihoods (N, 5)
        # 模型的 forward 返回的是 (y_star, likelihoods) 
        if isinstance(outputs, dict):
            probs = outputs["logits"]
        elif isinstance(outputs, tuple):
            probs = outputs[1]
        else:
            probs = outputs


        expected_scores = torch.sum(probs * score_tensor, dim=1)
        
        final_scores = torch.clamp(expected_scores.round(), 1, 5).int()
        
        for json_key, score in zip(ids, final_scores.tolist()):
            all_results.append({
                "id": json_key, 
                "prediction": int(score)
            })

print("\n手动推理完成。")

with open(OUTPUT_JSONL_PATH, 'w', encoding='utf-8') as f:
    for result in all_results:
        f.write(json.dumps(result) + '\n')

print(f"✅ 所有预测结果已保存到 {OUTPUT_JSONL_PATH}")

Model is on: cuda:0
Using Test/Inference file: /kaggle/input/semeval/data/dev.json
✅ 创建了 588 个有序分类样本 (Mode: predict)

***** 开始批量手动推理 (期望值预测) *****


Inferencing:   5%|▌         | 1/19 [00:00<00:12,  1.40it/s]

--- Y* Stats ---
Y* Mean: -0.6147
Y* Std. Dev.: 0.9131
----------------


Inferencing:  11%|█         | 2/19 [00:01<00:11,  1.44it/s]

--- Y* Stats ---
Y* Mean: -0.2981
Y* Std. Dev.: 0.7266
----------------


...

--- Y* Stats ---
Y* Mean: -0.0917
Y* Std. Dev.: 0.7622
----------------


Inferencing:  95%|█████████▍| 18/19 [00:12<00:00,  1.45it/s]

--- Y* Stats ---
Y* Mean: -0.5449
Y* Std. Dev.: 0.7065
----------------


Inferencing: 100%|██████████| 19/19 [00:12<00:00,  1.50it/s]

--- Y* Stats ---
Y* Mean: -0.7319
Y* Std. Dev.: 0.3552
----------------

手动推理完成。
✅ 所有预测结果已保存到 ./output_ordinal_regression/dev_predictions_mean.jsonl
